# External model — chat completion

OpenAI-compatible **`POST …/v1/chat/completions`** on the gateway host. The request uses **`stream: true`** (and **`Accept: text/event-stream`**) as typical for this API. The completion cell reads **`data:`** lines when present, parses **`choices[].delta`** (**`content`** / **`text`**), handles **`[DONE]`**, and falls back to a single JSON body when needed. If the response shape differs, a short **note** is printed.

**Prereq:** Python 3.9+ (stdlib: `json`, `ssl`, `urllib`).

**Credentials:** Same pattern as **`maas-validation-demo-with-key.ipynb`**: `MAAS_API_KEY` / `API_KEY` / **Demo quick swap** for the bearer token (mint with `POST …/maas-api/v1/api-keys` and subscription **`external-models-subscription`** as in your shell script). **`VERIFY_TLS`** from the environment (`1` / `true` for strict TLS).

**Two presets:** **Box A** = GPT-4o, **Box B** = Claude. Set **`ACTIVE_EXTERNAL`** in **Demo quick swap** to **`"gpt4o"`** or **`"claude"`**.

## Demo quick swap

Set **`ACTIVE_EXTERNAL`** (and optionally **`DEMO_API_KEY`** between takes). Run this cell, then **Load presets**.

| Variable | Effect |
|----------|--------|
| `ACTIVE_EXTERNAL` | **`"gpt4o"`** or **`"claude"`** — which preset to use. |
| `DEMO_API_KEY` | Non-empty → overrides `MAAS_API_KEY` / `API_KEY` env. |

In [ ]:
ACTIVE_EXTERNAL = "claude"  # or "gpt4o"
DEMO_API_KEY = ""

## Load presets

**HOST**, routes, and model ids are fixed here — only **`ACTIVE_EXTERNAL`** in **Demo quick swap** selects GPT-4o vs Claude.

The code cell also prints **copy-paste `curl`** lines (same URL, model, and JSON body as the completion cell) for your **terminal**.

In [ ]:
import json
import shlex
import os
import ssl
import urllib.error
import urllib.request

# Gateway host (same idea as HOST=maas.apps.... in curl)
HOST = "maas.apps.rosa.jland.li5b.p3.openshiftapps.com"

# --- Box A: GPT-4o ---
_PRESET_GPT4O = {
    "url": f"https://{HOST}/gpt-4o/v1/chat/completions",
    "model": "gpt-4o",
}

# --- Box B: Claude ---
_PRESET_CLAUDE = {
    "url": f"https://{HOST}/claude-sonnet-4-20250514/v1/chat/completions",
    "model": "claude-sonnet-4-20250514",
}

_presets = {"gpt4o": _PRESET_GPT4O, "claude": _PRESET_CLAUDE}
_ae = globals().get("ACTIVE_EXTERNAL", "gpt4o")
ACTIVE_EXTERNAL = (_ae.strip() if isinstance(_ae, str) and _ae.strip() else "gpt4o")
if ACTIVE_EXTERNAL not in _presets:
    raise SystemExit('ACTIVE_EXTERNAL must be "gpt4o" or "claude" (set in Demo quick swap)')
_sel = _presets[ACTIVE_EXTERNAL]
EXTERNAL_CHAT_COMPLETIONS_URL = _sel["url"]
EXTERNAL_MODEL_NAME = _sel["model"]

_ak = globals().get("DEMO_API_KEY", "")
if isinstance(_ak, str) and _ak.strip():
    API_KEY = _ak.strip()
else:
    API_KEY = (os.environ.get("MAAS_API_KEY") or os.environ.get("API_KEY") or "").strip()

VERIFY_TLS = os.environ.get("VERIFY_TLS", "").lower() in ("1", "true", "yes")

if not API_KEY:
    raise SystemExit("Set DEMO_API_KEY or MAAS_API_KEY / API_KEY.")

print("ACTIVE_EXTERNAL :", ACTIVE_EXTERNAL)
print("Model id          :", EXTERNAL_MODEL_NAME)
print("POST URL          :", EXTERNAL_CHAT_COMPLETIONS_URL)
print("VERIFY_TLS        :", VERIFY_TLS)
print("API key set       :", bool(API_KEY))


# Same JSON as the completion cell defaults (USER_MESSAGE / MAX_TOKENS)
_CURL_USER = "Say hello in one short sentence."
_curl_body = {
    "model": EXTERNAL_MODEL_NAME,
    "messages": [{"role": "user", "content": _CURL_USER}],
    "max_tokens": 256,
    "stream": True,
}
_curl_json = json.dumps(_curl_body, separators=(",", ":"))
_curl_k = "k" if not VERIFY_TLS else ""
print()
print("--- curl (terminal) ---")
print(f"export API_KEY={shlex.quote(API_KEY)}")
_curl_lines = [
    "curl -N -sS" + _curl_k + " " + chr(92),
    "  -H \"Authorization: Bearer $API_KEY\" " + chr(92),
    "  -H \"Content-Type: application/json\" " + chr(92),
    "  -d " + shlex.quote(_curl_json) + " " + chr(92),
    "  " + shlex.quote(EXTERNAL_CHAT_COMPLETIONS_URL),
]
print("\n".join(_curl_lines))
_curl_std_lines = [
    "stdbuf -oL curl -N -sS" + _curl_k + " " + chr(92),
    "  -H \"Authorization: Bearer $API_KEY\" " + chr(92),
    "  -H \"Content-Type: application/json\" " + chr(92),
    "  -d " + shlex.quote(_curl_json) + " " + chr(92),
    "  " + shlex.quote(EXTERNAL_CHAT_COMPLETIONS_URL),
]
print()
print("--- curl with stdbuf -oL (optional; GNU coreutils) ---")
print("\n".join(_curl_std_lines))
print()
print("# Omit export if API_KEY is already set.")
print("# Optional: curl -v for verbose headers.")


## Chat completion

Uses **`http.client`** with **`TCP_NODELAY`**, **`Accept-Encoding: identity`**, and **`Connection: close`**. Sends **`messages`** and **`stream: true`**, reads **`data:`** lines and **`choices[].delta`** when present, with fallback to a single JSON **`choices[].message.content`**. **`stdout`** is flushed after each write. If HTTP **200** yields no parsed text, a **note** is printed.

In [ ]:
USER_MESSAGE = "What Model am I using?"
MAX_TOKENS = 256

import http.client
import socket
import sys
import urllib.parse


try:
    sys.stdout.reconfigure(line_buffering=True)
except Exception:
    pass


def _flush_stdout():
    try:
        sys.stdout.flush()
        sys.stdout.buffer.flush()
    except Exception:
        sys.stdout.flush()


def _flush_print(s: str) -> None:
    """Write to stdout and flush."""
    sys.stdout.write(s)
    _flush_stdout()


def _ssl_ctx():
    ctx = ssl.create_default_context()
    if not VERIFY_TLS:
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
    return ctx


def _text_from_delta(delta):
    """OpenAI-style delta.content / delta.text, or list-shaped content blocks."""
    if not isinstance(delta, dict):
        return ""
    parts = []
    for key in ("content", "text"):
        v = delta.get(key)
        if isinstance(v, str) and v:
            parts.append(v)
        elif isinstance(v, list):
            for block in v:
                if isinstance(block, dict):
                    t = block.get("text") or block.get("content")
                    if isinstance(t, str):
                        parts.append(t)
                elif isinstance(block, str):
                    parts.append(block)
    return "".join(parts)


def _text_from_choice(ch):
    if not isinstance(ch, dict):
        return ""
    parts = []
    d = ch.get("delta")
    if isinstance(d, dict):
        parts.append(_text_from_delta(d))
    if isinstance(ch.get("text"), str) and ch["text"]:
        parts.append(ch["text"])
    msg = ch.get("message")
    if isinstance(msg, dict) and isinstance(msg.get("content"), str) and msg["content"]:
        parts.append(msg["content"])
    return "".join(parts)


def openai_chat_completion(
    url: str,
    *,
    api_key: str,
    model: str,
    user_message: str,
    max_tokens: int,
):
    """POST chat completions; print assistant text to stdout. Returns (http_status, error_or_none, format_note).
    On success, error is None. format_note is set if HTTP 200 but no text matched expected OpenAI-style parsing.
    On failure, http_status may be None if no HTTP response (e.g. TLS/DNS)."""
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": user_message}],
        "max_tokens": max_tokens,
        "stream": True,
    }
    req_body = json.dumps(payload).encode("utf-8")
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "Accept": "text/event-stream",
        "Accept-Encoding": "identity",
        "Connection": "close",
    }
    u = urllib.parse.urlsplit(url)
    if u.scheme.lower() not in ("http", "https"):
        raise SystemExit("URL must be http or https")
    host = u.hostname
    if not host:
        raise SystemExit("URL must include a host")
    port = u.port or (443 if u.scheme.lower() == "https" else 80)
    path = u.path or "/"
    if u.query:
        path += "?" + u.query
    ctx = _ssl_ctx()
    total_chars = 0
    last_usage = None
    conn = None
    try:
        if u.scheme.lower() == "https":
            conn = http.client.HTTPSConnection(host, port, context=ctx, timeout=300)
        else:
            conn = http.client.HTTPConnection(host, port, timeout=300)
        conn.connect()
        if conn.sock:
            conn.sock.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
        conn.request("POST", path, body=req_body, headers=headers)
        resp = conn.getresponse()
        status = resp.status
        err_body = None
        if status >= 400:
            err_body = resp.read().decode("utf-8", errors="replace")
            if not err_body.strip():
                err_body = f"HTTP {status}"
            print("HTTP", status, err_body[:2000])
            return (status, err_body, None)
        resp_body = b""
        while True:
            line = resp.readline()
            if not line:
                break
            resp_body += line
            s = line.decode("utf-8", errors="replace").strip()
            if not s or s.startswith(":"):
                continue
            if not s.startswith("data:"):
                continue
            payload_line = s[len("data:") :].lstrip()
            if payload_line == "[DONE]":
                break
            try:
                obj = json.loads(payload_line)
            except json.JSONDecodeError:
                continue
            if isinstance(obj, dict) and "usage" in obj:
                last_usage = obj.get("usage")
            for ch in obj.get("choices") or []:
                t = _text_from_choice(ch)
                if t:
                    _flush_print(t)
                    total_chars += len(t)
            if isinstance(obj, dict) and not obj.get("choices"):
                td = obj.get("delta")
                if isinstance(td, dict):
                    t = _text_from_delta(td)
                    if t:
                        _flush_print(t)
                        total_chars += len(t)
        if total_chars == 0 and resp_body.strip():
            try:
                obj = json.loads(resp_body.decode("utf-8", errors="replace").strip())
            except json.JSONDecodeError:
                obj = None
            if isinstance(obj, dict):
                for ch in obj.get("choices") or []:
                    t = _text_from_choice(ch)
                    if t:
                        _flush_print(t)
                        total_chars += len(t)
    except http.client.HTTPException as e:
        msg = str(e)
        print("HTTP client error:", msg)
        return (None, msg, None)
    except OSError as e:
        msg = str(e)
        print("Error:", msg)
        return (None, msg, None)
    finally:
        if conn is not None:
            try:
                conn.close()
            except Exception:
                pass
    print()
    _flush_stdout()
    print("---")
    print("Finished. Approx printed chars:", total_chars)
    if last_usage:
        print("usage:", last_usage)
    format_note = None
    if status == 200 and total_chars == 0 and resp_body.strip():
        format_note = (
            "Note: HTTP 200 but no assistant text was parsed. Expected OpenAI-style "
            "`data:` lines with choices[].delta (content/text), or one JSON with choices[].message.content. "
            "If the response shape differs, inspect the raw body."
        )
        print(format_note)
    return (status, None, format_note)


http_status, req_err, shape_note = openai_chat_completion(
    EXTERNAL_CHAT_COMPLETIONS_URL,
    api_key=API_KEY,
    model=EXTERNAL_MODEL_NAME,
    user_message=USER_MESSAGE,
    max_tokens=MAX_TOKENS,
)
print("HTTP status:", http_status, "| error:" if req_err else "| ok", req_err or "")